# Final Project: Build a Machine Learning Pipeline for Airfoil Noise Prediction
Dataset: NASA Airfoil Self Noise (modified) — memprediksi SoundLevel berdasarkan parameter airfoil

In [1]:
def warn(*args, **kwargs):  # suppress warning biar output bersih
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

import findspark
findspark.init()  # daftarkan lokasi Spark ke Python

In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Final Project - Airfoil Noise Prediction")
    .master("local[*]")  # jalan lokal pakai semua core CPU
    .getOrCreate())

In [3]:
import urllib.request

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/datasets/NASA_airfoil_noise_raw.csv"
urllib.request.urlretrieve(url, "NASA_airfoil_noise_raw.csv")  # simpan CSV ke folder kerja
print("Download selesai")

Download selesai


In [4]:
df = spark.read.csv("NASA_airfoil_noise_raw.csv", header=True, inferSchema=True)
df.show(5)  # cek 5 baris pertama

+---------+-------------+-----------+------------------+-----------------------+----------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevel|
+---------+-------------+-----------+------------------+-----------------------+----------+
|      800|          0.0|     0.3048|              71.3|             0.00266337|   126.201|
|     1000|          0.0|     0.3048|              71.3|             0.00266337|   125.201|
|     1250|          0.0|     0.3048|              71.3|             0.00266337|   125.951|
|     1600|          0.0|     0.3048|              71.3|             0.00266337|   127.591|
|     2000|          0.0|     0.3048|              71.3|             0.00266337|   127.461|
+---------+-------------+-----------+------------------+-----------------------+----------+
only showing top 5 rows


In [5]:
rowcount1 = df.count()
print(rowcount1)

1522


In [6]:
df = df.dropDuplicates()
rowcount2 = df.count()
print(rowcount2)

1503


In [7]:
df = df.dropna()
rowcount3 = df.count()
print(rowcount3)

1499


In [8]:
df = df.withColumnRenamed("SoundLevel", "SoundLevelDecibels")  # rename kolom target

In [9]:
df.write.mode("overwrite").parquet("NASA_airfoil_noise_cleaned.parquet")  # simpan hasil cleaning

In [10]:
print("Part 1 - Evaluation")
print("Total rows = ", rowcount1)
print("Total rows after dropping duplicate rows = ", rowcount2)
print("Total rows after dropping duplicate rows and rows with null values = ", rowcount3)
print("New column name = ", df.columns[-1])

import os
print("NASA_airfoil_noise_cleaned.parquet exists :", os.path.isdir("NASA_airfoil_noise_cleaned.parquet"))

Part 1 - Evaluation
Total rows =  1522
Total rows after dropping duplicate rows =  1503
Total rows after dropping duplicate rows and rows with null values =  1499
New column name =  SoundLevelDecibels
NASA_airfoil_noise_cleaned.parquet exists : True


In [11]:
df = spark.read.parquet("NASA_airfoil_noise_cleaned.parquet")
rowcount4 = df.count()  # jawaban Quiz Q6
print(rowcount4)

1499


In [12]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["Frequency", "AngleOfAttack", "ChordLength", "FreeStreamVelocity", "SuctionSideDisplacement"],
    outputCol="features"
)  # gabungkan semua kolom fitur jadi 1 kolom vektor

In [13]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")  # normalisasi skala fitur


In [14]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="scaledFeatures", labelCol="SoundLevelDecibels")  # target prediksi

In [15]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[assembler, scaler, lr])  # urutan: assemble -> scale -> model

In [16]:
(trainingData, testingData) = df.randomSplit([0.7, 0.3], seed=42)  # 70% train, 30% test

In [17]:
pipelineModel = pipeline.fit(trainingData)  # latih model pakai data training

In [18]:
print("Part 2 - Evaluation")
print("Total rows = ", rowcount4)

ps = [str(x).split("_")[0] for x in pipeline.getStages()]
print("Pipeline Stage 1 = ", ps[0])
print("Pipeline Stage 2 = ", ps[1])
print("Pipeline Stage 3 = ", ps[2])

print("Label column = ", lr.getLabelCol())

Part 2 - Evaluation
Total rows =  1499
Pipeline Stage 1 =  VectorAssembler
Pipeline Stage 2 =  StandardScaler
Pipeline Stage 3 =  LinearRegression
Label column =  SoundLevelDecibels


In [19]:
predictions = pipelineModel.transform(testingData)  # prediksi pakai data test

from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(labelCol="SoundLevelDecibels", predictionCol="prediction")

In [20]:
mse = evaluator.evaluate(predictions, {evaluator.metricName: "mse"})
print(mse)

24.99766625502418


In [21]:
mae = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
print(mae)

3.9136790958812044


In [22]:
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})
print(r2)

0.4959688408974623


In [23]:
print("Part 3 - Evaluation")
print("Mean Squared Error = ", round(mse,2))
print("Mean Absolute Error = ", round(mae,2))
print("R Squared = ", round(r2,2))

lrModel = pipelineModel.stages[-1]
print("Intercept = ", round(lrModel.intercept,2))

Part 3 - Evaluation
Mean Squared Error =  25.0
Mean Absolute Error =  3.91
R Squared =  0.5
Intercept =  132.88


In [24]:
pipelineModel.write().overwrite().save("Final_Project")  # simpan model

from pyspark.ml import PipelineModel
loadedPipelineModel = PipelineModel.load("Final_Project")  # load ulang

In [25]:
predictions = loadedPipelineModel.transform(testingData)
predictions.select("SoundLevelDecibels", "prediction").show(5)

+------------------+------------------+
|SoundLevelDecibels|        prediction|
+------------------+------------------+
|           128.679|122.59722914376778|
|            133.42|127.37968204568838|
|           119.146|130.34077425074506|
|           116.074|131.11016975113537|
|           134.319|127.12627360125096|
+------------------+------------------+
only showing top 5 rows


In [26]:
print("Part 4 - Evaluation")

loadedmodel = loadedPipelineModel.stages[-1]
totalstages = len(loadedPipelineModel.stages)
inputcolumns = loadedPipelineModel.stages[0].getInputCols()

print("Number of stages in the pipeline = ", totalstages)
for i,j in zip(inputcolumns, loadedmodel.coefficients):
    print(f"Coefficient for {i} is {round(j,4)}")

Part 4 - Evaluation
Number of stages in the pipeline =  3
Coefficient for Frequency is -3.9906
Coefficient for AngleOfAttack is -2.2881
Coefficient for ChordLength is -3.3269
Coefficient for FreeStreamVelocity is 1.4832
Coefficient for SuctionSideDisplacement is -2.0551


In [27]:
spark.stop()